In [1]:
import sys
from pathlib import Path
sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\examples')
from _offline import enforce_offline, blocked_attempts
enforce_offline()
import agentic_systems as ag
assert ag.__version__ == '2.1.3'
assert 'site-packages' in Path(ag.__file__).parts


# Runtime y Framework

Provider configura dónde se ejecuta un modelo; framework configura el loop del Agent. Backend configura su flujo.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [2]:
import agentic_systems as ag


In [3]:
@ag.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@ag.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = ag.skill(name="arithmetic", tools=[increment, double])
increase = ag.Action("increase", "increment")
multiply = ag.Action("multiply", "double")

In [4]:
for name in ("native", "langgraph", "openai-agents", "strands"):
    agent = ag.Agent(
        name="operator", runtime=ag.runtime(provider="python-runtime"),
        framework=ag.framework(name), skills=(arithmetic,),
    )
    agent.compile()  # prepara sin instanciar ni llamar al proveedor
    result = agent.run({"tool": "increment", "input": {"value": 3}})
    assert result.ok and result.data["value"] == 4
    print(name, result.data)

native {'value': 4, 'ok': True, 'tool': 'increment'}


langgraph {'value': 4, 'ok': True, 'tool': 'increment'}


openai-agents {'value': 4}


strands {'value': 4}


El mismo constructor Agent sirve para un flujo o para un actor con runtime explícito. Para incluir razonamiento en un flujo, coloca ese Agent como uno de sus pasos. No se infiere un proveedor ni se permite fallback automático. Las siguientes configuraciones **se declaran y no se ejecutan**: no acreditan compatibilidad live.

Los límites internos del actor se declaran en runtime. Limits limita el flujo completo. framework recibe opciones nativas mediante agent_kwargs y run_kwargs; las claves del ag no pueden sobrescribirse. Tener un SDK instalado o configurado no acredita un proveedor live.

In [5]:
configured = ag.runtime(provider='bedrock-runtime',model='explicit-configured-only',region='us-east-1',endpoint='https://fixture.invalid',api_key='fixture-secret',timeout_s=30,max_retries=1,max_tool_calls=3,max_turns=8,max_tokens=128,backoff_s=0.1,max_concurrency=2)
assert 'fixture-secret' not in str(configured.inspect())
print(configured.inspect())
options = ag.framework('openai-agents',agent_kwargs={'handoff_description':'configured-only'},run_kwargs={'context':{'tenant':'fixture'}})
print(options.inspect())

{'provider': 'bedrock-runtime', 'model': 'explicit-configured-only', 'region': 'us-east-1', 'endpoint': 'https://fixture.invalid', 'api_key_configured': True, 'timeout_s': 30, 'max_retries': 1, 'max_tool_calls': 3, 'max_turns': 8, 'max_tokens': 128, 'backoff_s': 0.1, 'max_concurrency': 2}
{'name': 'openai-agents', 'agent_kwargs': {'handoff_description': 'str'}, 'run_kwargs': {'context': 'dict'}}


In [6]:
configured_only = {
    name: ag.runtime(provider=name)
    for name in ("openai-runtime", "ollama-runtime", "bedrock-runtime", "vllm-runtime")
}
assert set(configured_only) == {"openai-runtime", "ollama-runtime", "bedrock-runtime", "vllm-runtime"}
print({name: "configurado; no ejecutado" for name in configured_only})

{'openai-runtime': 'configurado; no ejecutado', 'ollama-runtime': 'configurado; no ejecutado', 'bedrock-runtime': 'configurado; no ejecutado', 'vllm-runtime': 'configurado; no ejecutado'}


In [7]:
assert blocked_attempts() == 0